# Predict 1-Year US Stock Returns from Fundamentals

Thin Kaggle notebook for EDA and running the repository pipeline. Keep reusable logic in `src/` and CLI scripts.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

KAGGLE_DATA = Path('/kaggle/input/predict-1-year-us-stock-returns-from-fundamentals')
LOCAL_DATA = ROOT / 'data' / 'raw'
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT / 'outputs'

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = OUTPUT_DIR / 'submission.csv'

TRAIN_PATH, TEST_PATH, SUBMISSION_PATH

In [ ]:
import pandas as pd

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train.shape, test.shape

In [ ]:
train.head()

In [ ]:
train['return_pct'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
train.groupby('start_year')['return_pct'].agg(['count', 'mean', 'median', 'std']).round(4)

In [ ]:
from stock_returns.features import make_feature_frame
from stock_returns.validation import get_time_split

train_mask, valid_mask = get_time_split(train)
X_train = make_feature_frame(train.loc[train_mask])
X_valid = make_feature_frame(train.loc[valid_mask], fit_columns=list(X_train.columns))
X_train.shape, X_valid.shape

In [ ]:
from stock_returns.train import train_validation_pipeline

bundle = train_validation_pipeline(TRAIN_PATH, output_dir=OUTPUT_DIR, include_optional=True)
bundle['metrics']['ensemble']

In [ ]:
from stock_returns.predict import make_submission

submission = make_submission(TRAIN_PATH, TEST_PATH, SUBMISSION_PATH, include_optional=True)
submission.head(), submission.shape